In [6]:
import pandas as pd
import numpy as np
import os
import time
from tqdm.auto import tqdm
import duckdb
import sys

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du

importlib.reload(du)

<module 'utils.data_preprocessing' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/data_preprocessing.py'>

In [2]:
# Initialize notebook

# get parent dict
parent = os.path.dirname(os.getcwd())

# set up duckdb connection
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

In [ ]:
# load fut-data into memory, as it is used across all symbols
fut_dict = {}

for date in tqdm(du.SAMPLE_DATES, desc="Dates"):
    fut_df = con.execute(f"""
        SELECT "Timestamp", "Timestamp_Europe/Berlin", "MicroPrice_tick-based_10_1s" AS MicroPrice
        FROM read_csv('{parent}/data/raw/FUT_DAX Futures/FUT_DAX Futures_{date}.csv.gz')
    """).df()

    # resample_to_regular_grid samples onto a regular 100ms grid (floored, forward-filled)
    # Filter on the integer grid Timestamp (the ffilled Berlin column is stale on filled rows)
    # Both sides use the same grid, so the later backward merge_asof is an exact, leak-free per-bucket join.
    fut_df = du.resample_to_regular_grid(fut_df, "Timestamp", "100ms")
    fut_df = du.filter_trading_hours(fut_df, ts_col="Timestamp")

    fut = du.compute_feature_target_matrix(
        df=fut_df,
        ts_col='Timestamp',
        target_cols=[],
        feature_cols=['MicroPrice'],
        horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-5s', '-2s', '-1s', '-100ms'])

    fut_dict[date] = fut

In [ ]:
cols = ["Timestamp", "Timestamp_Europe/Berlin", "L1-BidPrice", "L1-AskPrice", "Side",
        "L1-QImb", "MidPrice", "MidPriceQW", "MidPriceCQW"]

symbols = [s for s in du.SYMBOLS if s != 'FUT_DAX Futures']
for symbol in tqdm(symbols, desc="Symbols"):

    os.makedirs(f'{parent}/data/processed/{symbol}', exist_ok=True)

    for date in tqdm(du.SAMPLE_DATES, desc="Dates", leave=False):
        path = f'{parent}/data/raw/CS_{symbol}/CS_{symbol}_{date}.csv.gz'

        group = con.execute(f"""
            SELECT {", ".join(f'"{c}"' for c in cols)}, "MicroPrice_tick-based_10_1s" AS MicroPrice
            FROM read_csv('{path}')
        """).df()

        group['TransactionPrice'] = du.compute_transaction_price(group)

        # TransactionPrice needs event-level Side, so it comes first.
        # The resampler then keeps the last state per 100ms bucket and ffills.
        # Filter on the integer grid Timestamp (the ffilled Berlin column is stale on filled rows).
        filtered = du.resample_to_regular_grid(group, "Timestamp", "100ms")
        filtered = du.filter_trading_hours(filtered, ts_col="Timestamp").copy()

        featured = du.compute_feature_target_matrix(
            df=filtered,
            ts_col='Timestamp',
            target_cols=du.PRICE_MEASURES,
            feature_cols=['L1-QImb', 'MicroPrice'],
            horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-5s', '-2s', '-1s', '-100ms',
                        '100ms', '1s', '2s', '5s', '15s', '30s', '1m', '2.5m', '5m'])

        # merge_asof requires both sides sorted on the key (Timestamp, int ns).
        # Both come out of resample_to_regular_grid / compute_feature_target_matrix sorted, so this is only a safety net.
        featured = featured.sort_values("Timestamp").reset_index(drop=True)

        merged = pd.merge_asof(
            featured,
            fut_dict[date].drop(columns="Timestamp_Europe/Berlin"),
            on="Timestamp",
            direction="backward",
            suffixes=("_LOB", "_FUT"))

        merged = merged.drop(columns=['TransactionPrice', 'MidPrice', 'MidPriceQW', 'MidPriceCQW', 'MicroPrice_FUT', 'MicroPrice_LOB',
                                        'Timestamp_Europe/Berlin', 'L1-BidPrice', 'L1-AskPrice', 'Side'])
        merged.to_parquet(f'{parent}/data/processed/{symbol}/{date}.parquet')

NameError: name 'du' is not defined

In [ ]:
test = pd.read_parquet(f"{parent}/data/processed/Adidas/2023-01-02.parquet")
test

In [ ]:
test.columns